<a href="https://www.kaggle.com/code/mrafraim/dl-day-49-cnn-fastapi-integration?scriptVersionId=314782554" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Day 49: CNN + FastAPI Integration

Welcome to Day 49!

What You’ll Learn Today:

1. How ML models are actually exposed via APIs
2. File upload handling in FastAPI
3. Connecting inference pipeline to API safely
4. Image preprocessing in server context
5. Real /predict endpoint design
6. Testing with real image input

If you found this notebook helpful, your **<b style="color:skyblue;">UPVOTE</b>** would be greatly appreciated! It helps others discover the work and supports continuous improvement.

---

# What You Are Building

This is NOT a model notebook anymore.

You are building a:<br>

👉 ML Prediction Service

**Flow:**

Client (Image Upload)<br>
↓<br>
FastAPI Endpoint (/predict)<br>
↓<br>
Image Decode (bytes → PIL)<br>
↓<br>
Preprocessing (strict)<br>
↓<br>
CNN Model Inference<br>
↓<br>
Response (JSON)

# HARD RULES

Inference service must NEVER contain:

❌ training loop  
❌ optimizer  
❌ loss function  
❌ data augmentation  
❌ backward()

**ONLY allowed:**

✔ load model  
✔ preprocess input  
✔ forward pass  
✔ return prediction  

# Imports
```python
# FastAPI core framework to build web APIs
from fastapi import FastAPI, File, UploadFile  

# PIL (Python Imaging Library) for opening and processing images
from PIL import Image  

# PyTorch core library for tensor operations and model inference
import torch  

# torchvision transforms for preprocessing images (resize, normalize, tensor conversion, etc.)
import torchvision.transforms as transforms  

# io module to handle image byte streams (used when reading uploaded files)
import io  
```

# Initialize FastAPI App

```python
app = FastAPI()
```

# Load Model

Model must NOT reload per request.
That kills performance.

```python
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CNN().to(device)

model.load_state_dict(
    torch.load("best_model.pth", map_location=device)
)

model.eval()

```

- `model.eval()` = deterministic behavior
- prevents dropout randomness
- fixes batchnorm behavior

Preprocessing Pipeline (MUST MATCH TRAINING)

```python
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
```

If preprocessing differs even slightly:<br>
→ model performance collapses silently

This is the #1 production ML failure source

# Clean Inference Function

```python
def predict_image(image: Image.Image):

    # Apply preprocessing
    image = transform(image)

    # Add batch dimension
    image = image.unsqueeze(0).to(device)

    # Inference only (no gradients)
    with torch.no_grad():
        outputs = model(image)
        pred = outputs.argmax(dim=1).item()

    return pred
```

This function is:

✔ independent  
✔ reusable  
✔ testable  
✔ API-agnostic  

This is how real ML services are structured.

# /predict Endpoint (File Upload API)

```python
@app.post("/predict")
async def predict(file: UploadFile = File(...)):

    # Step 1: Read uploaded file bytes
    image_bytes = await file.read()

    # Step 2: Convert bytes → PIL image
    image = Image.open(io.BytesIO(image_bytes))

    # Step 3: Run inference pipeline
    pred_class = predict_image(image)

    return {
        "prediction_index": pred_class
    }
```

What this endpoint does

1. Accepts image upload
2. Decodes binary data
3. Runs model inference
4. Returns structured JSON response

# Testing via /docs (Critical Tool)
**Open in browser:**

http://127.0.0.1:8000/docs

Why /docs matters

✔ no frontend needed  
✔ instant API testing  
✔ validates input schema  
✔ debugs file upload easily  

# Real Production Failure Points
1. Model loaded per request → latency explosion
2. Wrong preprocessing → silent wrong predictions
3. Missing `eval()` → random outputs
4. Large images → memory crash
5. No error handling → API crash

# Mental Model

You are NOT building a model.

You are building:

ML Service = API + Model + Preprocessing + Inference

Layers:

1. API layer (FastAPI)
2. Data layer (image decoding)
3. Processing layer (transforms)
4. Model layer (CNN)
5. Output layer (JSON response)

# Key Takeaways from Day 49

- ML models become useful only when served via APIs
- Preprocessing consistency is critical
- Model must load once, not per request
- Inference must be deterministic
- /docs is your debugging interface
- You are now building ML systems, not models

---

<p style="text-align:center; color:skyblue; font-size:18px;">
© 2026 Mostafizur Rahman
</p>
